# 05 — Deterministic reproducibility of fold assignment

Fold assignment in `FoldPlanner` is purely a function of the configuration (`n_folds`, azimuth extent, range extent); it draws no random numbers. This notebook confirms that rebuilding the planner repeatedly, and across the `seed` field of `CrossValidationConfig`, yields byte-identical fold plans.

Modules exercised: `configuration.cross_validation_config.CrossValidationConfig`, `pipelines.cross_validation_pipeline.folds.FoldPlanner`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## A canonical serialisation of a fold plan

We reduce each plan to a tuple of its block assignments and `CropRegion` tuples so two plans can be compared exactly.

In [ ]:
from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner

def serialise(planner):
    rows = []
    for plan in planner.plans():
        splits = {name: tuple(r.as_tuple() for r in plan.split_regions.regions(name)) for name in ("train", "val", "test")}
        rows.append((plan.fold_index, plan.test_block, plan.val_block, tuple(plan.train_blocks), splits["train"], splits["val"], splits["test"]))
    return tuple(rows)

def build(seed):
    config                     = CrossValidationConfig()
    config.seed                = seed
    config.folds.n_folds       = 7
    config.folds.azimuth_start = 0
    config.folds.azimuth_end   = 700
    return FoldPlanner(config, range_start=0, range_end=80)

## Rebuild many times and compare against the first plan

In [ ]:
reference = serialise(build(seed=0))

repeat_matches = [serialise(build(seed=0)) == reference for _ in range(20)]
print("20 rebuilds identical:", all(repeat_matches))

seed_matches = {seed: serialise(build(seed=seed)) == reference for seed in (0, 1, 7, 42, 123)}
for seed, match in seed_matches.items():
    print(f"seed {seed:3d}: identical to reference -> {match}")

## Visualise stability as a determinism matrix

Twelve independent rebuilds form the columns; each cell reports whether that rebuild matches the reference. An all-green column-by-row grid is the visual confirmation of determinism.

In [ ]:
import matplotlib.colors as mcolors

n_rebuilds = 12
rebuilds   = [serialise(build(seed=0)) for _ in range(n_rebuilds)]
match_grid = np.array([[1 if rb == reference else 0 for rb in rebuilds]])

cmap    = mcolors.ListedColormap(["#d96c6c", "#7bb37b"])
fig, ax = plt.subplots(figsize=(9, 1.6))
ax.imshow(match_grid, cmap=cmap, aspect="auto", vmin=0, vmax=1)

ax.set_yticks([0])
ax.set_yticklabels(["plan == reference"])
ax.set_xticks(range(n_rebuilds))
ax.set_xlabel("rebuild index")
ax.set_title("Determinism of fold assignment across rebuilds")
plt.show()

## Expected visual outcome

All twenty repeat rebuilds report identical plans, every tested seed matches the reference, and the determinism matrix is uniformly green. This confirms fold assignment is fully deterministic and seed-independent.